# Fit on train, score on train

Open `lib/pipelines.py`. Each pipeline scales numeric columns and one-hot encodes district, plan, and aisle before classification. `cross_val_score` refits on train rows only.

The pipeline fits the scaler on the same rows as the classifier, so preprocessing does not use test data.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    if (candidate / "app.py").is_file() and (candidate / "lib" / "schema.py").is_file():
        root = candidate
        break
    nested = candidate / "01-supervised-learning"
    if (nested / "app.py").is_file() and (nested / "lib" / "schema.py").is_file():
        root = nested
        break
if root is None:
    raise FileNotFoundError(
        "Start Jupyter from the repo folder or from 01-supervised-learning."
    )
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("cwd", here)
print("module", ROOT)


In [ ]:
import pandas as pd

customers = pd.read_csv(ROOT / "data" / "fresh_and_good_customers.csv")
print(customers.shape)
customers.head()


In [ ]:
from frame import features_and_target
from split import split_customers

train, validation, test = split_customers(customers)
print("train", len(train), "validation", len(validation), "test", len(test))
X_train, y_train = features_and_target(train)
X_val, y_val = features_and_target(validation)
X_test, y_test = features_and_target(test)
print("churned is a label, not a feature:", "churned" not in X_train.columns)


In [ ]:
from sklearn.model_selection import cross_val_score

from pipelines import make_logistic_pipeline

logistic = make_logistic_pipeline()
logistic.fit(X_train, y_train)
logistic_cv = cross_val_score(
    logistic, X_train, y_train, cv=3, scoring="roc_auc", n_jobs=1
)
print("logistic cv mean", round(float(logistic_cv.mean()), 3))


### Exercises

Fit a forest pipeline the same way. `make_forest_pipeline` is in `pipelines`. Use `cross_val_score` on `X_train` and `y_train` only, with `cv=3` and `scoring="roc_auc"`. Store the mean in `forest_cv_mean`.

Then score the fitted logistic model on the validation rows. `roc_auc_score` needs the labels and column 1 of `predict_proba`. Store that number in `logistic_val_auc`.


In [ ]:
from sklearn.metrics import roc_auc_score

from pipelines import make_forest_pipeline

forest = None
forest_cv_mean = None
logistic_val_auc = None


In [ ]:
if forest_cv_mean is None or logistic_val_auc is None:
    raise NotImplementedError("Finish the forest CV exercise and the logistic validation AUC.")
print("forest cv mean", round(float(forest_cv_mean), 3))
print("logistic validation AUC", round(float(logistic_val_auc), 3))


## Checkpoint

Cross-validation here is only a train-set check. Notebook 04 compares models on the validation rows. If you are behind at 13:35, skip the forest exercise, keep the logistic printout, and move on.
